# 01 - Analise de Dados Experimentais
> Metodos estatisticos, caracterizacao de materiais (XRD, MEV, espectroscopia)

**Modulo:** `11_engfis_tecnico` | **Feito com:** [Claude](https://claude.ai) (Anthropic)

---


## Setup e Configuracao

Importamos as bibliotecas necessarias e configuramos o cliente da API Claude.

In [ ]:
import os
from anthropic import Anthropic

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import curve_fit

client = Anthropic()

HAIKU = "claude-haiku-4-20250414"
SONNET = "claude-sonnet-4-20250514"

def ask(prompt, model=SONNET, system="", max_tokens=4096):
    """Envia um prompt para Claude e retorna a resposta."""
    messages = [{"role": "user", "content": prompt}]
    kwargs = {"model": model, "max_tokens": max_tokens, "messages": messages}
    if system:
        kwargs["system"] = system
    response = client.messages.create(**kwargs)
    return response.content[0].text

print("Setup completo!")

---
## Escolhendo Metodos Estatisticos

Em fisica experimental, a escolha do metodo estatistico correto e fundamental para
validar resultados. Vamos gerar dados sinteticos de medidas de resistividade e
pedir ao Claude para recomendar os testes estatisticos apropriados.

In [ ]:
# Gerar dados sinteticos de resistividade (ohm.cm) para 3 amostras
np.random.seed(42)

amostra_A = np.random.normal(loc=1.72e-6, scale=0.05e-6, size=30)  # Cobre puro
amostra_B = np.random.normal(loc=1.78e-6, scale=0.08e-6, size=30)  # Cobre com impurezas
amostra_C = np.random.normal(loc=2.65e-6, scale=0.10e-6, size=30)  # Aluminio

print("Amostra A (Cobre puro):")
print(f"  Media = {np.mean(amostra_A):.4e} ohm.cm")
print(f"  Desvio padrao = {np.std(amostra_A, ddof=1):.4e} ohm.cm")
print(f"  N = {len(amostra_A)}")
print()
print("Amostra B (Cobre com impurezas):")
print(f"  Media = {np.mean(amostra_B):.4e} ohm.cm")
print(f"  Desvio padrao = {np.std(amostra_B, ddof=1):.4e} ohm.cm")
print(f"  N = {len(amostra_B)}")
print()
print("Amostra C (Aluminio):")
print(f"  Media = {np.mean(amostra_C):.4e} ohm.cm")
print(f"  Desvio padrao = {np.std(amostra_C, ddof=1):.4e} ohm.cm")
print(f"  N = {len(amostra_C)}")

In [ ]:
# Visualizar as distribuicoes
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, data, nome, cor in zip(
    axes,
    [amostra_A, amostra_B, amostra_C],
    ["Amostra A (Cu puro)", "Amostra B (Cu+imp)", "Amostra C (Al)"],
    ["steelblue", "coral", "seagreen"]
):
    ax.hist(data, bins=10, color=cor, edgecolor="black", alpha=0.7)
    ax.axvline(np.mean(data), color="red", linestyle="--", label=f"Media")
    ax.set_xlabel("Resistividade (ohm.cm)")
    ax.set_ylabel("Frequencia")
    ax.set_title(nome)
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Perguntar ao Claude sobre metodos estatisticos apropriados
prompt_stats = f"""Sou estudante de engenharia fisica na UFRGS. Fiz medidas de resistividade
eletrica em 3 amostras metalicas (30 medicoes cada). Os dados resumidos sao:

Amostra A (Cobre puro): media = {np.mean(amostra_A):.4e} ohm.cm, std = {np.std(amostra_A, ddof=1):.4e}
Amostra B (Cobre com impurezas): media = {np.mean(amostra_B):.4e} ohm.cm, std = {np.std(amostra_B, ddof=1):.4e}
Amostra C (Aluminio): media = {np.mean(amostra_C):.4e} ohm.cm, std = {np.std(amostra_C, ddof=1):.4e}

Quero saber:
1. Que testes estatisticos devo usar para comparar as amostras A e B? (mesma liga)
2. Que teste devo usar para comparar as 3 amostras simultaneamente?
3. Como verificar se meus dados seguem distribuicao normal?
4. Qual o papel do teste chi-quadrado nesse contexto?

Explique cada metodo, quando usar, e as hipoteses (H0, H1) de forma objetiva."""

system_stats = """Voce e um professor de fisica experimental. Responda de forma clara e
tecnica, usando notacao estatistica. Mantenha a resposta concisa e pratica."""

resposta_stats = ask(prompt_stats, system=system_stats)
print(resposta_stats)

In [ ]:
# Aplicar os testes recomendados pelo Claude

# Teste de normalidade (Shapiro-Wilk)
print("=== Teste de Normalidade (Shapiro-Wilk) ===")
for nome, data in [("A", amostra_A), ("B", amostra_B), ("C", amostra_C)]:
    stat, p = stats.shapiro(data)
    print(f"  Amostra {nome}: W={stat:.4f}, p={p:.4f} {'(Normal)' if p > 0.05 else '(Nao normal)'}")

print()

# Teste t de Student (A vs B)
print("=== Teste t (Amostra A vs B) ===")
t_stat, t_p = stats.ttest_ind(amostra_A, amostra_B)
print(f"  t = {t_stat:.4f}, p = {t_p:.4f}")
print(f"  {'Rejeita H0: medias sao diferentes' if t_p < 0.05 else 'Nao rejeita H0: medias nao diferem significativamente'}")

print()

# ANOVA (comparar as 3 amostras)
print("=== ANOVA (3 amostras) ===")
f_stat, f_p = stats.f_oneway(amostra_A, amostra_B, amostra_C)
print(f"  F = {f_stat:.4f}, p = {f_p:.4e}")
print(f"  {'Rejeita H0: pelo menos uma media difere' if f_p < 0.05 else 'Nao rejeita H0'}")

print()

# Teste Chi-quadrado para aderencia a distribuicao esperada
print("=== Teste Chi-quadrado (aderencia - Amostra A) ===")
observed, bin_edges = np.histogram(amostra_A, bins=8)
# Calcular frequencias esperadas (distribuicao normal)
cdf_vals = stats.norm.cdf(bin_edges, loc=np.mean(amostra_A), scale=np.std(amostra_A, ddof=1))
expected = np.diff(cdf_vals) * len(amostra_A)
# Agrupar bins com frequencia esperada < 5
chi2, chi_p = stats.chisquare(observed, f_exp=expected)
print(f"  chi2 = {chi2:.4f}, p = {chi_p:.4f}")

---
## Interpretando Difratogramas (XRD)

A difracao de raios X (XRD) e uma tecnica fundamental para identificar fases cristalinas.
Vamos simular um difratograma e usar Claude para interpretar os picos e calcular
parametros de rede usando a Lei de Bragg: $n\lambda = 2d\sin\theta$.

In [ ]:
# Simular difratograma de uma amostra de cobre (FCC)
# Picos principais do Cu: (111), (200), (220), (311), (222)

def gaussian_peak(two_theta, center, amplitude, sigma):
    """Gera um pico gaussiano."""
    return amplitude * np.exp(-0.5 * ((two_theta - center) / sigma)**2)

# Angulos 2theta experimentais do Cu (radiacao Cu-Ka, lambda=1.5406 A)
cu_peaks = {
    "(111)": {"2theta": 43.30, "intensity": 100, "sigma": 0.15},
    "(200)": {"2theta": 50.45, "intensity": 46,  "sigma": 0.15},
    "(220)": {"2theta": 74.13, "intensity": 20,  "sigma": 0.18},
    "(311)": {"2theta": 89.93, "intensity": 17,  "sigma": 0.20},
    "(222)": {"2theta": 95.14, "intensity": 5,   "sigma": 0.20},
}

# Gerar espectro completo
two_theta = np.linspace(20, 110, 5000)
intensity = np.random.normal(0, 0.5, len(two_theta))  # ruido de fundo
intensity = np.clip(intensity, 0, None)  # remover valores negativos

# Adicionar fundo decrescente
background = 2.0 * np.exp(-0.01 * (two_theta - 20))
intensity += background

# Adicionar picos
for hkl, params in cu_peaks.items():
    intensity += gaussian_peak(two_theta, params["2theta"], params["intensity"], params["sigma"])

# Plotar difratograma
plt.figure(figsize=(12, 5))
plt.plot(two_theta, intensity, "b-", linewidth=0.5)
for hkl, params in cu_peaks.items():
    plt.annotate(hkl, xy=(params["2theta"], params["intensity"] + 3),
                 ha="center", fontsize=9, color="red")
plt.xlabel(r"$2\theta$ (graus)")
plt.ylabel("Intensidade (u.a.)")
plt.title("Difratograma XRD - Amostra Metalica")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Preparar dados dos picos para enviar ao Claude
peak_data = "Posicoes dos picos observados no difratograma (radiacao Cu-Ka, lambda=1.5406 A):\n"
peak_data += "2theta (graus) | Intensidade relativa (%)\n"
peak_data += "-" * 45 + "\n"
for hkl, params in cu_peaks.items():
    peak_data += f"  {params['2theta']:.2f}          | {params['intensity']}\n"

prompt_xrd = f"""Sou estudante de engenharia fisica. Obtive o seguinte difratograma de raios X
de uma amostra metalica desconhecida:

{peak_data}

Por favor:
1. Identifique o material e a estrutura cristalina com base nos picos.
2. Indexe os picos (atribua indices de Miller hkl).
3. Calcule o parametro de rede 'a' usando a Lei de Bragg para cada pico.
4. Mostre as formulas usadas e o calculo passo a passo para o primeiro pico.
5. Qual o parametro de rede medio e sua incerteza?

Use a formula para FCC: d_hkl = a / sqrt(h^2 + k^2 + l^2)
e a Lei de Bragg: n*lambda = 2*d*sin(theta)"""

system_xrd = """Voce e um cristalografo experiente. Mostre os calculos detalhados
usando notacao cientifica. Responda de forma didatica."""

resposta_xrd = ask(prompt_xrd, system=system_xrd)
print(resposta_xrd)

In [ ]:
# Verificar calculando parametro de rede com Python
lambda_xrd = 1.5406  # Angstrom (Cu-Ka)

miller_indices = {
    "(111)": (1, 1, 1),
    "(200)": (2, 0, 0),
    "(220)": (2, 2, 0),
    "(311)": (3, 1, 1),
    "(222)": (2, 2, 2),
}

print("Calculo do parametro de rede (a) para cada pico:")
print(f"{'Pico':<8} {'2theta':>8} {'d (A)':>10} {'a (A)':>10}")
print("-" * 40)

a_values = []
for hkl, (h, k, l) in miller_indices.items():
    two_theta_deg = cu_peaks[hkl]["2theta"]
    theta_rad = np.radians(two_theta_deg / 2)
    d = lambda_xrd / (2 * np.sin(theta_rad))  # Lei de Bragg
    a = d * np.sqrt(h**2 + k**2 + l**2)       # Relacao para FCC
    a_values.append(a)
    print(f"{hkl:<8} {two_theta_deg:>8.2f} {d:>10.4f} {a:>10.4f}")

a_medio = np.mean(a_values)
a_std = np.std(a_values, ddof=1)
print(f"\nParametro de rede medio: a = {a_medio:.4f} +/- {a_std:.4f} A")
print(f"Valor de referencia do Cu: a = 3.6149 A")
print(f"Erro relativo: {abs(a_medio - 3.6149)/3.6149 * 100:.2f}%")

---
## Analise de Imagens MEV

Microscopia Eletronica de Varredura (MEV/SEM) produz imagens de alta resolucao
da superficie de materiais. Vamos gerar uma imagem sintetica de microestrutura
e usar Claude para ajudar a escrever codigo de analise de imagem.

In [ ]:
# Gerar imagem sintetica tipo MEV (graos policristalinos com poros)
from scipy.ndimage import gaussian_filter, label

np.random.seed(123)
size = 256

# Criar textura de graos usando ruido suavizado
noise = np.random.rand(size, size)
smooth_noise = gaussian_filter(noise, sigma=8)

# Normalizar para 0-255 (simular niveis de cinza do MEV)
img = ((smooth_noise - smooth_noise.min()) / (smooth_noise.max() - smooth_noise.min()) * 200 + 30)

# Adicionar poros (regioes escuras circulares)
pore_mask = np.zeros((size, size), dtype=bool)
n_pores = 15
for _ in range(n_pores):
    cx, cy = np.random.randint(20, size-20, 2)
    radius = np.random.randint(3, 12)
    y, x = np.ogrid[-cx:size-cx, -cy:size-cy]
    mask = x**2 + y**2 <= radius**2
    pore_mask[mask] = True

img[pore_mask] = np.random.uniform(10, 40, np.sum(pore_mask))

# Adicionar contornos de grao
edges = np.abs(np.gradient(smooth_noise, axis=0)) + np.abs(np.gradient(smooth_noise, axis=1))
edges_norm = edges / edges.max()
grain_boundaries = edges_norm > 0.03
img[grain_boundaries] = img[grain_boundaries] * 0.5

# Adicionar ruido tipo MEV
img += np.random.normal(0, 5, (size, size))
img = np.clip(img, 0, 255).astype(np.uint8)

plt.figure(figsize=(8, 8))
plt.imshow(img, cmap="gray")
plt.title("Imagem MEV Sintetica - Microestrutura Policristalina")
plt.colorbar(label="Nivel de cinza")
plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Pedir ao Claude para ajudar com analise da imagem MEV
prompt_mev = """Preciso analisar uma imagem de MEV (Microscopia Eletronica de Varredura)
de uma amostra ceramica policristalina. A imagem tem 256x256 pixels em escala de cinza.

A imagem mostra:
- Graos com diferentes niveis de cinza (contraste de orientacao)
- Poros escuros (regioes com nivel de cinza < 50)
- Contornos de grao como linhas mais escuras entre os graos

Escreva um codigo Python completo usando numpy e scipy para:
1. Segmentar os poros por limiarizacao (threshold) e calcular a porosidade percentual
2. Identificar e contar os poros individuais usando rotulacao conectada
3. Calcular a distribuicao de tamanho dos poros (area de cada poro em pixels)
4. Gerar um histograma da distribuicao de tamanho de graos

Use scipy.ndimage para as operacoes de imagem. A variavel 'img' ja contem a imagem
como array numpy uint8. Adicione comentarios explicativos."""

system_mev = """Voce e especialista em analise de imagens de microscopia. Escreva
codigo Python limpo e funcional. Use apenas numpy, scipy.ndimage e matplotlib."""

resposta_mev = ask(prompt_mev, system=system_mev)
print(resposta_mev)

In [ ]:
# Implementar a analise de poros sugerida pelo Claude
from scipy.ndimage import label, binary_fill_holes, binary_opening

# 1. Segmentacao por limiarizacao
threshold = 50
pore_binary = img < threshold

# Limpeza morfologica (remover ruido)
pore_clean = binary_opening(pore_binary, structure=np.ones((3, 3)))

# 2. Rotulacao conectada
labeled_pores, num_pores = label(pore_clean)
print(f"Numero de poros identificados: {num_pores}")

# 3. Calcular area de cada poro
pore_areas = []
for i in range(1, num_pores + 1):
    area = np.sum(labeled_pores == i)
    if area > 5:  # ignorar artefatos muito pequenos
        pore_areas.append(area)

total_pixels = img.shape[0] * img.shape[1]
porosity = np.sum(pore_clean) / total_pixels * 100

print(f"Porosidade: {porosity:.2f}%")
print(f"Area media dos poros: {np.mean(pore_areas):.1f} pixels")
print(f"Area mediana dos poros: {np.median(pore_areas):.1f} pixels")

# 4. Visualizacao
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(img, cmap="gray")
axes[0].set_title("Imagem original")
axes[0].axis("off")

axes[1].imshow(labeled_pores, cmap="nipy_spectral")
axes[1].set_title(f"Poros segmentados (N={len(pore_areas)})")
axes[1].axis("off")

axes[2].hist(pore_areas, bins=15, color="steelblue", edgecolor="black")
axes[2].set_xlabel("Area do poro (pixels)")
axes[2].set_ylabel("Frequencia")
axes[2].set_title("Distribuicao de tamanho dos poros")
axes[2].axvline(np.mean(pore_areas), color="red", linestyle="--", label="Media")
axes[2].legend()

plt.tight_layout()
plt.show()

---
## Analise Espectroscopica

A espectroscopia e usada para identificar composicao quimica e estrutura molecular.
Vamos criar espectros sinteticos de FTIR e UV-Vis e pedir ao Claude para
interpretar os picos observados.

In [ ]:
# Simular espectro FTIR de um polimero (ex: PET - polietileno tereftalato)
wavenumber = np.linspace(4000, 400, 3000)  # cm^-1

# Baseline (transmitancia ~95%)
transmittance = 95 + np.random.normal(0, 0.3, len(wavenumber))

# Picos de absorcao tipicos do PET
ftir_peaks = {
    "O-H stretch":       {"center": 3440, "depth": 25, "width": 120},
    "C-H aromatic":      {"center": 3060, "depth": 10, "width": 30},
    "C-H stretch":       {"center": 2960, "depth": 15, "width": 40},
    "C=O ester":         {"center": 1720, "depth": 55, "width": 25},
    "C=C aromatic":      {"center": 1580, "depth": 20, "width": 20},
    "C-O-C stretch":     {"center": 1245, "depth": 40, "width": 30},
    "C-O stretch":       {"center": 1100, "depth": 35, "width": 25},
    "C-H aromatic oop":  {"center": 725,  "depth": 30, "width": 20},
}

for name, params in ftir_peaks.items():
    peak = params["depth"] * np.exp(-0.5 * ((wavenumber - params["center"]) / params["width"])**2)
    transmittance -= peak

transmittance = np.clip(transmittance, 0, 100)

# Plotar espectro FTIR
plt.figure(figsize=(12, 5))
plt.plot(wavenumber, transmittance, "b-", linewidth=0.6)
plt.xlabel("Numero de onda (cm$^{-1}$)")
plt.ylabel("Transmitancia (%)")
plt.title("Espectro FTIR - Amostra Polimerica Desconhecida")
plt.xlim(4000, 400)
plt.grid(True, alpha=0.3)

# Marcar picos principais
for name, params in ftir_peaks.items():
    if params["depth"] > 20:
        plt.annotate(f"{params['center']} cm$^{{-1}}$",
                     xy=(params["center"], 95 - params["depth"]),
                     xytext=(params["center"], 95 - params["depth"] - 8),
                     fontsize=7, ha="center", color="red")

plt.tight_layout()
plt.show()

In [ ]:
# Formatar dados espectrais para enviar ao Claude
peak_list = "Picos de absorcao observados no espectro FTIR:\n"
peak_list += f"{'Numero de onda (cm-1)':<25} {'Intensidade relativa':<20}\n"
peak_list += "-" * 50 + "\n"

# Ordenar por profundidade (intensidade)
sorted_peaks = sorted(ftir_peaks.items(), key=lambda x: x[1]["depth"], reverse=True)
for name, params in sorted_peaks:
    intensidade = "Forte" if params["depth"] > 35 else "Media" if params["depth"] > 15 else "Fraca"
    peak_list += f"  {params['center']:<25} {intensidade} ({params['depth']}%)\n"

prompt_ftir = f"""Analise este espectro FTIR de uma amostra polimerica desconhecida:

{peak_list}

Por favor:
1. Identifique os grupos funcionais associados a cada pico.
2. Baseado no conjunto de picos, identifique o polimero mais provavel.
3. Explique por que o pico em 1720 cm-1 e tao intenso.
4. O que a presenca de picos aromaticos indica sobre a estrutura?
5. Ha alguma evidencia de degradacao ou contaminacao nos picos observados?"""

system_ftir = """Voce e espectroscopista com experiencia em caracterizacao de polimeros.
Seja preciso nas atribuicoes de picos e na identificacao do material."""

resposta_ftir = ask(prompt_ftir, system=system_ftir)
print(resposta_ftir)

In [ ]:
# Simular espectro UV-Vis de uma solucao com nanoparticulas de ouro
wavelength = np.linspace(300, 800, 2000)  # nm

# Baseline
absorbance = 0.05 + 0.1 * np.exp(-0.005 * (wavelength - 300))  # scattering background

# Pico de ressonancia plasmon de superficie (SPR) de AuNPs (~520 nm)
spr_peak = 1.2 * np.exp(-0.5 * ((wavelength - 522) / 25)**2)
absorbance += spr_peak

# Absorcao interbanda do Au (< 400 nm)
interband = 0.4 * np.exp(-0.02 * (wavelength - 300))
absorbance += interband

# Ruido
absorbance += np.random.normal(0, 0.005, len(wavelength))

plt.figure(figsize=(10, 5))
plt.plot(wavelength, absorbance, "b-", linewidth=0.8)
plt.xlabel("Comprimento de onda (nm)")
plt.ylabel("Absorbancia")
plt.title("Espectro UV-Vis - Solucao de Nanoparticulas")
plt.axvline(522, color="red", linestyle="--", alpha=0.5, label="SPR ~522 nm")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Perguntar ao Claude sobre o espectro UV-Vis
prompt_uv = f"""Obtive um espectro UV-Vis de uma solucao coloidal. Os dados mostram:
- Pico intenso de absorbancia em ~522 nm (absorbancia = {np.max(absorbance):.3f})
- FWHM do pico principal ~ 50 nm
- Absorcao crescente abaixo de 400 nm
- Fundo de scattering suave

1. Que tipo de nanoparticulas provavelmente produziu este espectro?
2. O que a posicao do pico SPR indica sobre o tamanho das particulas?
3. O que o FWHM indica sobre a dispersao de tamanho?
4. Como a forma das particulas afetaria o espectro?"""

resposta_uv = ask(prompt_uv, system="Voce e especialista em nanomateriais e plasmonica.")
print(resposta_uv)

---
## Propagacao de Incertezas

Em medidas experimentais, a propagacao de incertezas e essencial. Vamos usar
Claude para derivar formulas de propagacao para cadeias de medida complexas.

In [ ]:
# Exemplo: Calcular modulo de Young a partir de ensaio de tracao
# E = (F/A) / (dL/L0) = F * L0 / (A * dL)
# onde F=forca, A=area da secao, L0=comprimento inicial, dL=deformacao

prompt_incerteza = """Preciso calcular o modulo de Young (E) a partir de um ensaio de tracao.
A formula e:

E = (F * L0) / (A * delta_L)

onde:
- F = forca aplicada = (500.0 +/- 2.5) N
- L0 = comprimento inicial = (100.0 +/- 0.5) mm
- A = area da secao transversal = pi * d^2 / 4, com d = (5.00 +/- 0.05) mm
- delta_L = deformacao = (0.250 +/- 0.005) mm

Por favor:
1. Derive a formula completa de propagacao de incertezas para E.
2. Note que A depende de d, entao precisa propagar a incerteza de d para A primeiro.
3. Calcule a derivada parcial de E em relacao a cada variavel.
4. Calcule o valor de E e sua incerteza absoluta e relativa.
5. Qual variavel contribui mais para a incerteza total?

Mostre todas as etapas do calculo."""

system_incerteza = """Voce e professor de fisica experimental. Mostre a derivacao
completa usando o metodo de derivadas parciais (GUM). Use notacao clara."""

resposta_incerteza = ask(prompt_incerteza, system=system_incerteza)
print(resposta_incerteza)

In [ ]:
# Verificacao numerica da propagacao de incertezas

# Valores medidos
F, u_F = 500.0, 2.5        # N
L0, u_L0 = 100.0, 0.5      # mm
d, u_d = 5.00, 0.05         # mm
dL, u_dL = 0.250, 0.005     # mm

# Area e sua incerteza
A = np.pi * d**2 / 4
u_A = np.pi * d / 2 * u_d  # dA/dd = pi*d/2

# Modulo de Young
E = (F * L0) / (A * dL)  # em N/mm^2 = MPa
E_GPa = E / 1000  # converter para GPa

# Propagacao de incertezas (derivadas parciais)
# u_E/E = sqrt((u_F/F)^2 + (u_L0/L0)^2 + (u_A/A)^2 + (u_dL/dL)^2)
rel_F = (u_F / F)**2
rel_L0 = (u_L0 / L0)**2
rel_A = (u_A / A)**2
rel_dL = (u_dL / dL)**2

u_E_rel = np.sqrt(rel_F + rel_L0 + rel_A + rel_dL)
u_E = E * u_E_rel
u_E_GPa = u_E / 1000

print("=== Calculo do Modulo de Young ===")
print(f"Area: A = {A:.4f} +/- {u_A:.4f} mm^2")
print(f"Modulo de Young: E = {E:.1f} MPa = {E_GPa:.2f} GPa")
print(f"Incerteza absoluta: u(E) = {u_E:.1f} MPa = {u_E_GPa:.2f} GPa")
print(f"Incerteza relativa: {u_E_rel*100:.2f}%")
print(f"\nResultado: E = ({E_GPa:.2f} +/- {u_E_GPa:.2f}) GPa")
print()
print("=== Contribuicoes individuais ===")
total = rel_F + rel_L0 + rel_A + rel_dL
print(f"  Forca (F):       {rel_F/total*100:.1f}%")
print(f"  Comprimento (L0): {rel_L0/total*100:.1f}%")
print(f"  Area (A->d):     {rel_A/total*100:.1f}%")
print(f"  Deformacao (dL): {rel_dL/total*100:.1f}%")

# Grafico de contribuicoes
labels = ["Forca (F)", "Comprimento (L0)", "Area (d)", "Deformacao (dL)"]
contributions = [rel_F/total*100, rel_L0/total*100, rel_A/total*100, rel_dL/total*100]
colors = ["steelblue", "coral", "seagreen", "gold"]

plt.figure(figsize=(8, 5))
plt.bar(labels, contributions, color=colors, edgecolor="black")
plt.ylabel("Contribuicao para incerteza total (%)")
plt.title("Balanco de Incertezas - Modulo de Young")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

---
## Ajuste de Curvas e Modelos

Ajustar modelos a dados experimentais e uma das tarefas mais comuns em fisica.
Vamos usar Claude para identificar o modelo correto e implementar o ajuste
usando `scipy.optimize.curve_fit`.

In [ ]:
# Gerar dados ruidosos seguindo um modelo de decaimento exponencial
# Exemplo: decaimento radioativo ou relaxacao de tensao

np.random.seed(77)

# Parametros verdadeiros
A_true = 150.0    # amplitude
tau_true = 3.5     # constante de tempo
C_true = 10.0      # offset (background)

# Dados experimentais
t_data = np.linspace(0, 20, 50)
y_true = A_true * np.exp(-t_data / tau_true) + C_true
y_noise = np.random.normal(0, 5, len(t_data))  # ruido gaussiano
y_data = y_true + y_noise
y_err = np.full_like(y_data, 5.0)  # barras de erro

plt.figure(figsize=(10, 5))
plt.errorbar(t_data, y_data, yerr=y_err, fmt="ko", markersize=4,
             capsize=3, label="Dados experimentais")
plt.xlabel("Tempo (s)")
plt.ylabel("Sinal (u.a.)")
plt.title("Dados Experimentais - Decaimento Desconhecido")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Pedir ao Claude para sugerir o modelo de ajuste
# Formatamos uma amostra dos dados para enviar
data_sample = "t (s)    | Sinal (u.a.)\n"
data_sample += "-" * 30 + "\n"
indices = [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 49]
for i in indices:
    if i < len(t_data):
        data_sample += f"{t_data[i]:6.2f}   | {y_data[i]:8.2f}\n"

prompt_fit = f"""Tenho dados experimentais de um processo de decaimento temporal.
Uma amostra dos dados:

{data_sample}

O sinal comeca alto e decai ate estabilizar em um valor constante (~10).
Incerteza estimada: +/- 5 u.a. em cada ponto.

Por favor:
1. Sugira o modelo matematico mais apropriado para ajustar estes dados.
2. Escreva o codigo Python usando scipy.optimize.curve_fit para fazer o ajuste.
3. Inclua estimativas iniciais razoaveis para os parametros (p0).
4. Mostre como extrair as incertezas dos parametros ajustados a partir da
   matriz de covariancia.
5. Calcule o chi-quadrado reduzido para avaliar a qualidade do ajuste.
6. Plote os dados com barras de erro e a curva ajustada.

As variaveis t_data, y_data e y_err ja estao definidas."""

system_fit = """Voce e fisico experimental com experiencia em analise de dados.
Escreva codigo Python funcional e bem documentado."""

resposta_fit = ask(prompt_fit, system=system_fit)
print(resposta_fit)

In [ ]:
# Implementar o ajuste

# Modelo: decaimento exponencial com offset
def decay_model(t, A, tau, C):
    """y = A * exp(-t/tau) + C"""
    return A * np.exp(-t / tau) + C

# Estimativas iniciais
p0 = [y_data[0] - y_data[-1], 3.0, y_data[-1]]

# Ajuste com barras de erro
popt, pcov = curve_fit(decay_model, t_data, y_data, p0=p0, sigma=y_err, absolute_sigma=True)

# Incertezas dos parametros
perr = np.sqrt(np.diag(pcov))

A_fit, tau_fit, C_fit = popt
u_A, u_tau, u_C = perr

print("=== Resultados do Ajuste ===")
print(f"A   = {A_fit:.2f} +/- {u_A:.2f} u.a.")
print(f"tau = {tau_fit:.3f} +/- {u_tau:.3f} s")
print(f"C   = {C_fit:.2f} +/- {u_C:.2f} u.a.")

# Chi-quadrado reduzido
residuals = y_data - decay_model(t_data, *popt)
chi2 = np.sum((residuals / y_err)**2)
ndf = len(t_data) - len(popt)  # graus de liberdade
chi2_red = chi2 / ndf

print(f"\nChi-quadrado reduzido: {chi2_red:.3f} (ideal ~ 1.0)")
print(f"Graus de liberdade: {ndf}")

print(f"\n=== Comparacao com valores verdadeiros ===")
print(f"A:   verdadeiro = {A_true}, ajustado = {A_fit:.2f}")
print(f"tau: verdadeiro = {tau_true}, ajustado = {tau_fit:.3f}")
print(f"C:   verdadeiro = {C_true}, ajustado = {C_fit:.2f}")

In [ ]:
# Grafico final: dados + ajuste + residuos
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), height_ratios=[3, 1],
                                sharex=True, gridspec_kw={"hspace": 0.05})

# Painel superior: dados e ajuste
t_smooth = np.linspace(0, 20, 500)
ax1.errorbar(t_data, y_data, yerr=y_err, fmt="ko", markersize=4,
             capsize=3, label="Dados experimentais")
ax1.plot(t_smooth, decay_model(t_smooth, *popt), "r-", linewidth=2,
         label=f"Ajuste: A={A_fit:.1f}, tau={tau_fit:.2f}s, C={C_fit:.1f}")
ax1.fill_between(t_smooth,
                 decay_model(t_smooth, A_fit-u_A, tau_fit, C_fit),
                 decay_model(t_smooth, A_fit+u_A, tau_fit, C_fit),
                 color="red", alpha=0.1, label="Banda de confianca (1 sigma)")
ax1.set_ylabel("Sinal (u.a.)")
ax1.set_title(f"Ajuste Exponencial - $\\chi^2_{{red}}$ = {chi2_red:.3f}")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Painel inferior: residuos
ax2.errorbar(t_data, residuals, yerr=y_err, fmt="ko", markersize=4, capsize=3)
ax2.axhline(0, color="red", linewidth=1)
ax2.axhline(5, color="gray", linestyle="--", alpha=0.5)
ax2.axhline(-5, color="gray", linestyle="--", alpha=0.5)
ax2.set_xlabel("Tempo (s)")
ax2.set_ylabel("Residuos")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Exercicios

### Exercicio 1 - Analise Estatistica
Gere dados sinteticos de dureza Vickers para duas ligas diferentes (50 medidas cada).
Use Claude para determinar se as medias sao estatisticamente diferentes e qual
teste usar. Implemente o teste e interprete o resultado.

### Exercicio 2 - XRD de Liga Binaria
Crie um difratograma simulado com picos de duas fases cristalinas (ex: Fe-BCC e
Fe3C). Envie os dados ao Claude e peca para identificar ambas as fases e estimar
a fracao volumetrica relativa usando as intensidades dos picos.

### Exercicio 3 - Analise de Imagem Avancada
Modifique o codigo de analise MEV para:
- Calcular a circularidade dos poros (4*pi*area/perimetro^2)
- Classificar os poros em "esfericos" e "irregulares"
- Peca ao Claude para explicar o significado fisico da forma dos poros

### Exercicio 4 - Espectroscopia Raman
Simule um espectro Raman de grafeno (picos G, D, 2D) com diferentes razoes
de intensidade I(D)/I(G). Peca ao Claude para interpretar a qualidade do
grafeno baseado nessas razoes.

### Exercicio 5 - Propagacao Completa
Calcule a condutividade termica usando o metodo de flash laser:
k = (rho * Cp * alpha), onde alpha = L^2 / (pi^2 * t_half).
Peca ao Claude para derivar a propagacao de incertezas completa para k
e identifique a variavel dominante.

### Exercicio 6 - Ajuste Nao-Linear Complexo
Gere dados que seguem uma funcao Voigt (convolucao de Gaussiana com Lorentziana),
comum em espectroscopia. Use Claude para ajudar a implementar o ajuste e
comparar com ajustes puramente Gaussianos e Lorentzianos.

---
## Proximos Passos

Neste notebook aprendemos a usar Claude como assistente para analise de dados
experimentais em engenharia fisica. Os proximos topicos incluem:

1. **Simulacao computacional** - Usar Claude para ajudar a configurar simulacoes
   de elementos finitos (FEM) e dinamica molecular (MD).

2. **Automacao de laboratorio** - Integrar Claude com instrumentacao via Python
   para coleta e analise automatizada de dados.

3. **Machine learning para materiais** - Usar Claude para ajudar a construir
   modelos preditivos de propriedades de materiais.

4. **Redacao cientifica** - Usar Claude para revisar textos, gerar legendas
   de figuras e formatar referencias.

### Dicas para uso eficaz do Claude em fisica experimental

- **Seja especifico**: inclua unidades, condicoes experimentais e contexto.
- **Formate os dados**: tabelas e listas facilitam a interpretacao pelo modelo.
- **Use system prompts**: defina o papel do Claude (cristalografo, espectroscopista).
- **Verifique sempre**: Claude pode errar em calculos numericos; confira com codigo.
- **Itere**: refine suas perguntas com base nas respostas anteriores.

---
*Notebook criado para o curso de Engenharia Fisica - UFRGS*